# COMP6223 CW2 - Intel Image Classification
Dataset structure expected: `data/seg_train/seg_train/<class>/` and `data/seg_test/seg_test/<class>/`

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install tqdm
!pip install scikit-learn

In [ ]:
import os
import glob
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import confusion_matrix, accuracy_score, average_precision_score, classification_report

BASE_DIR     = os.getcwd()
DATA_DIR     = os.path.join(BASE_DIR, 'intel_image_classification')
CLASSES      = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
IMG_SIZE     = (128, 128)
CODEBOOK_K   = 500
DSIFT_STEP   = 8
DSIFT_SCALES = [8, 16]
PCA_DIMS     = 128
SPM_LEVELS   = [1, 2, 4]
SIGMA        = 1.0

## Load Dataset

In [ ]:
def load_split(root):
    images, labels = [], []
    for label_idx, cls in enumerate(CLASSES):
        for p in sorted(glob.glob(os.path.join(root, cls, '*.jpg'))):
            try:
                img = Image.open(p).convert('RGB').resize(IMG_SIZE)
                images.append(np.array(img, dtype=np.uint8))
                labels.append(label_idx)
            except Exception:
                pass
    return np.array(images), np.array(labels)

X_train, y_train = load_split(os.path.join(DATA_DIR, 'seg_train', 'seg_train'))
X_test,  y_test  = load_split(os.path.join(DATA_DIR, 'seg_test',  'seg_test'))
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## Canny Edge Detection & Harris Corners (Traditional CV)

In [ ]:
def to_gray(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

def gaussian_smooth(gray, sigma=SIGMA):
    ksize = int(6 * sigma + 1) | 1
    return cv2.GaussianBlur(gray, (ksize, ksize), sigma)

def canny_edges(gray, low=50, high=150):
    return cv2.Canny(gaussian_smooth(gray), low, high)

def harris_corners(gray, k=0.04, threshold=0.01):
    resp = cv2.cornerHarris(np.float32(gray), 2, 3, k)
    resp = cv2.dilate(resp, None)
    return resp, int((resp > threshold * resp.max()).sum())

def extract_traditional_features(img_rgb):
    gray  = to_gray(img_rgb)
    edges = canny_edges(gray)
    _, nc = harris_corners(gray)
    H, W  = gray.shape
    mh, mw = H // 2, W // 2
    quads = [
        edges[:mh, :mw].mean() / 255, edges[:mh, mw:].mean() / 255,
        edges[mh:, :mw].mean() / 255, edges[mh:, mw:].mean() / 255,
    ]
    return np.array([edges.mean() / 255, nc / (H * W)] + quads, dtype=np.float32)

In [ ]:
trad_train = np.vstack([extract_traditional_features(img) for img in tqdm(X_train, desc='Train')])
trad_test  = np.vstack([extract_traditional_features(img) for img in tqdm(X_test,  desc='Test')])
print(f'Traditional feature shape: {trad_train.shape}')

## Dense SIFT Descriptors

In [ ]:
def dense_sift(img_rgb):
    gray = gaussian_smooth(to_gray(img_rgb))
    sift = cv2.SIFT_create()
    H, W = gray.shape
    all_desc, all_pos = [], []
    for patch_size in DSIFT_SCALES:
        r = patch_size // 2
        kps = [cv2.KeyPoint(float(x), float(y), float(patch_size))
               for y in range(r, H - r + 1, DSIFT_STEP)
               for x in range(r, W - r + 1, DSIFT_STEP)]
        if not kps:
            continue
        _, desc = sift.compute(gray, kps)
        if desc is not None:
            all_desc.append(desc)
            all_pos.append(np.array([[kp.pt[0], kp.pt[1]] for kp in kps], np.float32))
    if not all_desc:
        return np.zeros((1, 128), np.float32), np.zeros((1, 2), np.float32)
    return np.vstack(all_desc).astype(np.float32), np.vstack(all_pos).astype(np.float32)

In [ ]:
descs_train, pos_train = zip(*[dense_sift(img) for img in tqdm(X_train, desc='SIFT train')])
descs_test,  pos_test  = zip(*[dense_sift(img) for img in tqdm(X_test,  desc='SIFT test')])
descs_train, pos_train = list(descs_train), list(pos_train)
descs_test,  pos_test  = list(descs_test),  list(pos_test)

## Build Codebook (K-Means)

In [ ]:
rng = np.random.default_rng(42)
sampled = []
for d in descs_train:
    n = min(100, len(d))
    sampled.append(d[rng.choice(len(d), n, replace=False)])

codebook = MiniBatchKMeans(n_clusters=CODEBOOK_K, random_state=42, batch_size=4096, n_init=3)
codebook.fit(np.vstack(sampled).astype(np.float32))
print(f'Codebook: {codebook.n_clusters} visual words')

## Spatial Pyramid BoVW Encoding

In [ ]:
def _bovw_hist(word_ids, k):
    h = np.bincount(word_ids, minlength=k).astype(np.float32)
    return h / (np.linalg.norm(h) + 1e-7)

def encode_image_spm(desc, pos, img_shape):
    k        = codebook.n_clusters
    word_ids = codebook.predict(desc.astype(np.float32))
    H, W     = img_shape
    parts    = []
    for grid in SPM_LEVELS:
        if grid == 1:
            parts.append(_bovw_hist(word_ids, k))
        else:
            ch, cw = H / grid, W / grid
            for r in range(grid):
                for c in range(grid):
                    mask = ((pos[:, 1] >= r * ch) & (pos[:, 1] < (r + 1) * ch) &
                            (pos[:, 0] >= c * cw) & (pos[:, 0] < (c + 1) * cw))
                    ids = word_ids[mask]
                    parts.append(_bovw_hist(ids, k) if ids.size > 0 else np.zeros(k, np.float32))
    L = len(SPM_LEVELS)
    weights = np.concatenate([
        np.full(SPM_LEVELS[i] ** 2 * k,
                1.0 if SPM_LEVELS[i] == 1 else 2.0 ** (-(L - 1 - i)),
                dtype=np.float32)
        for i in range(L)
    ])
    return np.concatenate(parts) * weights

In [ ]:
shapes_train = [(img.shape[0], img.shape[1]) for img in X_train]
shapes_test  = [(img.shape[0], img.shape[1]) for img in X_test]

X_bovw_train = np.vstack([encode_image_spm(d, p, s)
                           for d, p, s in tqdm(zip(descs_train, pos_train, shapes_train), total=len(X_train))])
X_bovw_test  = np.vstack([encode_image_spm(d, p, s)
                           for d, p, s in tqdm(zip(descs_test,  pos_test,  shapes_test),  total=len(X_test))])
print(f'BoVW shape: {X_bovw_train.shape}')

## PCA

In [ ]:
pca = PCA(n_components=PCA_DIMS, whiten=True, random_state=42)
pca.fit(X_bovw_train)
print(f'Explained variance: {pca.explained_variance_ratio_.sum():.1%}')

X_feat_train = np.hstack([pca.transform(X_bovw_train).astype(np.float32), trad_train])
X_feat_test  = np.hstack([pca.transform(X_bovw_test).astype(np.float32),  trad_test])
print(f'Final feature dim: {X_feat_train.shape[1]}')

## Train SVM (One-vs-Rest, 5-fold CV)

In [ ]:
best_C, best_score = 0.01, 0.0
for C in [0.01, 0.1, 1.0, 10.0]:
    clf = Pipeline([('sc', StandardScaler()),
                    ('svm', OneVsRestClassifier(LinearSVC(C=C, max_iter=2000, random_state=42)))])
    scores = cross_val_score(clf, X_feat_train, y_train,
                             cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             scoring='accuracy', n_jobs=-1)
    print(f'C={C}  acc={scores.mean():.4f} ± {scores.std():.4f}')
    if scores.mean() > best_score:
        best_score, best_C = scores.mean(), C

print(f'Best C={best_C}')
clf = Pipeline([('sc', StandardScaler()),
                ('svm', OneVsRestClassifier(LinearSVC(C=best_C, max_iter=2000, random_state=42)))])
clf.fit(X_feat_train, y_train)

## Evaluate

In [ ]:
y_pred    = clf.predict(X_feat_test)
X_scaled  = clf.named_steps['sc'].transform(X_feat_test)
scores    = clf.named_steps['svm'].decision_function(X_scaled)
acc       = accuracy_score(y_test, y_pred)
y_bin     = label_binarize(y_test, classes=list(range(len(CLASSES))))
per_ap    = {cls: average_precision_score(y_bin[:, i], scores[:, i]) for i, cls in enumerate(CLASSES)}
map_score = np.mean(list(per_ap.values()))
cm        = confusion_matrix(y_test, y_pred)

print(f'Accuracy : {acc:.4f}')
print(f'mAP      : {map_score:.4f}')
print('\nPer-class AP:')
for cls, ap in per_ap.items():
    print(f'  {cls:12s}: {ap:.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=CLASSES))

## Confusion Matrix

In [ ]:
_, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set(xticks=range(len(CLASSES)), yticks=range(len(CLASSES)),
       xticklabels=CLASSES, yticklabels=CLASSES,
       xlabel='Predicted', ylabel='True', title='Confusion Matrix')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
thresh = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=9,
                color='white' if cm[i, j] > thresh else 'black')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## Sample Features (Original / Canny / Harris)

In [ ]:
selected = []
for cls_idx in range(len(CLASSES)):
    selected.extend(np.where(y_train == cls_idx)[0][:2])

_, axes = plt.subplots(len(selected), 3, figsize=(9, 3 * len(selected)))
for row, img_idx in enumerate(selected):
    img   = X_train[img_idx]
    gray  = to_gray(img)
    edges = canny_edges(gray)
    resp, _ = harris_corners(gray)
    corner_vis = img.copy()
    corner_vis[resp > 0.01 * resp.max()] = [255, 0, 0]
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"{CLASSES[y_train[img_idx]]}\nOriginal", fontsize=8)
    axes[row, 1].imshow(edges, cmap='gray')
    axes[row, 1].set_title('Canny Edges', fontsize=8)
    axes[row, 2].imshow(corner_vis)
    axes[row, 2].set_title('Harris Corners', fontsize=8)
    for ax in axes[row]:
        ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'sample_features.png'), dpi=150)
plt.show()